# Horarios Pico
**Proyecto:** Reserva Inteligente de Restaurantes — Etapa 3  
**Análisis:** Distribución de demanda por hora, día de semana y tipo de día  
**Fuente:** Data Warehouse Hive (`restaurant_dw`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("horarios_pico_notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.shuffle.partitions", "8")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE restaurant_dw")
print("Spark version:", spark.version)

In [ ]:
fact_pedido      = spark.table("fact_pedido")
fact_reservacion = spark.table("fact_reservacion")
dim_tiempo       = spark.table("dim_tiempo")
dim_restaurante  = spark.table("dim_restaurante")
dim_estado       = spark.table("dim_estado_pedido")

print(f"fact_pedido: {fact_pedido.count()} filas")
print(f"fact_reservacion: {fact_reservacion.count()} filas")

## 1. Demanda por hora del día

In [ ]:
df_hora = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante,  fact_pedido["id_restaurante"] == dim_restaurante["id"])
    .groupBy(
        dim_tiempo["hora"],
        dim_tiempo["es_hora_pico"],
        dim_tiempo["es_fin_semana"],
        dim_restaurante["nombre"].alias("restaurante"),
    )
    .agg(
        F.countDistinct(fact_pedido["id_pedido_origen"]).alias("total_pedidos"),
        F.countDistinct(fact_pedido["id_usuario"]).alias("clientes_unicos"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rest = Window.partitionBy("restaurante")
df_hora = df_hora.withColumn(
    "pct_pedidos_del_total",
    F.round(F.col("total_pedidos") * 100.0 / F.sum("total_pedidos").over(w_rest), 2)
).orderBy("restaurante", "hora")

df_hora.show(24, truncate=False)

## 2. Pico por día de semana × hora

In [ ]:
df_dia_hora = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante,  fact_pedido["id_restaurante"] == dim_restaurante["id"])
    .groupBy(
        dim_tiempo["dia_semana"],
        dim_tiempo["nombre_dia"],
        dim_tiempo["hora"],
        dim_tiempo["es_fin_semana"],
        dim_restaurante["nombre"].alias("restaurante"),
    )
    .agg(
        F.countDistinct(fact_pedido["id_pedido_origen"]).alias("total_pedidos"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rank = Window.partitionBy("restaurante", "dia_semana").orderBy(F.desc("total_pedidos"))
df_dia_hora = (
    df_dia_hora
    .withColumn("rank_hora_en_dia", F.rank().over(w_rank))
    .orderBy("restaurante", "dia_semana", "hora")
)

# Mostrar solo las horas top por día
df_dia_hora.filter(F.col("rank_hora_en_dia") <= 3).show(40, truncate=False)

## 3. Fin de semana vs días de semana

In [ ]:
df_tipo_dia = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante,  fact_pedido["id_restaurante"] == dim_restaurante["id"])
    .withColumn(
        "tipo_dia",
        F.when(F.col("es_fin_semana"), F.lit("Fin de semana"))
         .otherwise(F.lit("Día de semana"))
    )
    .groupBy(
        "tipo_dia",
        dim_restaurante["nombre"].alias("restaurante"),
        dim_tiempo["hora"],
    )
    .agg(
        F.countDistinct(fact_pedido["id_pedido_origen"]).alias("total_pedidos"),
        F.round(F.avg("subtotal"), 2).alias("ticket_promedio"),
        F.round(F.sum("subtotal"), 2).alias("ingresos_totales"),
    )
    .orderBy("restaurante", "tipo_dia", "hora")
)

df_tipo_dia.show(30, truncate=False)

## 4. Ocupación de mesas por hora

In [ ]:
df_ocupacion = (
    fact_reservacion
    .join(dim_tiempo,       fact_reservacion["id_tiempo"]      == dim_tiempo["id"])
    .join(dim_restaurante,  fact_reservacion["id_restaurante"] == dim_restaurante["id"])
    .filter(F.col("estado") == "reservada")
    .groupBy(
        dim_tiempo["hora"],
        dim_tiempo["es_fin_semana"],
        dim_restaurante["nombre"].alias("restaurante"),
    )
    .agg(
        F.count("*").alias("total_reservaciones"),
        F.round(F.avg("cant_personas"), 2).alias("personas_promedio"),
        F.round(F.avg("tasa_ocupacion"), 2).alias("ocupacion_promedio_pct"),
        F.round(F.avg("duracion_minutos"), 2).alias("duracion_promedio_min"),
    )
    .orderBy("restaurante", "hora")
)

df_ocupacion.show(24, truncate=False)

## 5. Guardar resultados

In [ ]:
df_hora.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_demanda_por_hora")
df_dia_hora.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_pico_dia_semana")
df_tipo_dia.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_fin_semana_vs_semana")
df_ocupacion.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_ocupacion_mesas_horaria")

print("✅ Resultados guardados en Hive.")
spark.stop()